# Notebook 40 — Token Streaming & Partial Results

This notebook demonstrates the streaming primitives in the Multigen SDK.

| Class | Role |
|---|---|
| `StreamToken` | A single streaming chunk (text, source, metadata) |
| `StreamingAgent` | Wraps an async-generator into a named streaming agent |
| `ParallelStreamer` | Fans out to N agents and interleaves their token streams |
| `StreamAggregator` | Merges raw async-generator streams into one ordered stream |
| `PartialResult` | Incremental result event emitted during execution |
| `PartialResultBus` | Pub/sub bus for partial result events |

## Setup

In [ ]:
import sys
sys.path.insert(0, '../sdk')

import asyncio
from multigen.streaming import (
    StreamToken, StreamingAgent, ParallelStreamer,
    StreamAggregator, PartialResult, PartialResultBus,
)

print('Streaming module loaded successfully')

## Section 1: StreamingAgent Basics

`StreamingAgent` wraps any `async def` generator into a named streaming agent.
You can either:
- Iterate tokens one by one with `.stream(ctx)`
- Collect the full text with `.run(ctx)`

In [ ]:
# Mock LLM stream: yields words one at a time with a small delay
async def mock_llm_stream(ctx):
    """Simulates an LLM streaming words."""
    sentence = ctx.get('prompt', 'Hello from the streaming agent!')
    words = sentence.split()
    for word in words:
        await asyncio.sleep(0.02)  # simulate token latency
        yield word + ' '

# Wrap with StreamingAgent
agent = StreamingAgent('story_writer', stream_fn=mock_llm_stream)
print(f'Agent created: {agent.name}')

In [ ]:
# Stream tokens one by one — each token arrives as soon as it is produced
ctx = {'prompt': 'The quick brown fox jumps over the lazy dog'}

print('Streaming tokens:')
print('-' * 50)

async def stream_demo():
    token_count = 0
    async for token in agent.stream(ctx):
        print(f'  token[{token_count:02d}]: {token.text!r:15s}  source={token.source}  final={token.is_final}')
        token_count += 1
    print(f'\nTotal tokens received: {token_count}')

await stream_demo()

In [ ]:
# .run() collects the full stream and returns a single string
async def collect_demo():
    full_text = await agent.run(ctx)
    print(f'Full text collected ({len(full_text)} chars):')
    print(f'  "{full_text.strip()}"')

await collect_demo()

In [ ]:
# StreamingAgent also supports an on_token callback
tokens_seen = []

def token_logger(token: StreamToken):
    tokens_seen.append(token.text)

agent_with_cb = StreamingAgent(
    'logged_writer',
    stream_fn=mock_llm_stream,
    on_token=token_logger,
)

async def callback_demo():
    await agent_with_cb.run({'prompt': 'Callbacks fire on every token'})
    print(f'Callback recorded {len(tokens_seen)} tokens')
    print(f'First 3: {tokens_seen[:3]}')

await callback_demo()

## Section 2: Parallel Streaming

`ParallelStreamer` fans out to N `StreamingAgent` instances and interleaves their
token streams. Tokens arrive as soon as *any* branch produces them — no waiting
for all branches to complete. Each token carries a `source` label.

This is useful for:
- Showing multiple model responses simultaneously
- Running debates or multi-perspective generation

In [ ]:
import random

# Three agents with different characteristic speeds and styles
async def fast_agent_stream(ctx):
    """Fast, terse responses."""
    words = ['Fast:', 'concise,', 'quick', 'answer.']
    for w in words:
        await asyncio.sleep(random.uniform(0.01, 0.03))
        yield w + ' '

async def medium_agent_stream(ctx):
    """Medium speed, balanced responses."""
    words = ['Medium:', 'balanced', 'and', 'thoughtful', 'response.']
    for w in words:
        await asyncio.sleep(random.uniform(0.03, 0.07))
        yield w + ' '

async def slow_agent_stream(ctx):
    """Slow, verbose responses."""
    words = ['Slow:', 'detailed,', 'comprehensive,', 'in-depth', 'analysis', 'here.']
    for w in words:
        await asyncio.sleep(random.uniform(0.05, 0.10))
        yield w + ' '

agent_fast   = StreamingAgent('fast_agent',   stream_fn=fast_agent_stream)
agent_medium = StreamingAgent('medium_agent', stream_fn=medium_agent_stream)
agent_slow   = StreamingAgent('slow_agent',   stream_fn=slow_agent_stream)

print('Three streaming agents created:', [a.name for a in [agent_fast, agent_medium, agent_slow]])

In [ ]:
# ParallelStreamer merges all three streams — tokens interleave by arrival time
ps = ParallelStreamer([agent_fast, agent_medium, agent_slow])

print('Interleaved tokens (source shown in brackets):')
print('-' * 60)

async def parallel_stream_demo():
    async for token in ps.stream({'topic': 'AI safety'}):
        print(f'  [{token.source:<15}] {token.text!r}')

await parallel_stream_demo()

In [ ]:
# ps.run() waits for all branches and returns {agent_name: full_text}
async def parallel_run_demo():
    results = await ps.run({'topic': 'AI safety'})
    print('Full text per branch:')
    for name, text in results.items():
        print(f'  {name:<15}: "{text.strip()}"')

await parallel_run_demo()

## Section 3: StreamAggregator

`StreamAggregator` is a lower-level primitive that accepts raw async generators
(not `StreamingAgent` instances). It merges them into a single token stream.

Use it when:
- You have generators from external libraries
- You want to compose arbitrary async iterators

In [ ]:
# Raw async generators — no StreamingAgent wrapper needed
async def finance_stream(ctx):
    for chunk in ['Revenue', 'grew', '12%', 'YoY. ']:
        await asyncio.sleep(0.03)
        yield chunk + ' '

async def tech_stream(ctx):
    for chunk in ['New', 'GPU', 'architecture', 'launched. ']:
        await asyncio.sleep(0.04)
        yield chunk + ' '

async def policy_stream(ctx):
    for chunk in ['Regulation', 'bill', 'passed', 'Senate. ']:
        await asyncio.sleep(0.02)
        yield chunk + ' '

print('Raw async generators defined')

In [ ]:
ctx = {'query': 'daily briefing'}

# StreamAggregator.add() accepts (label, async_generator)
agg = StreamAggregator()
agg.add('finance', finance_stream(ctx))
agg.add('tech',    tech_stream(ctx))
agg.add('policy',  policy_stream(ctx))

print('Merged stream from 3 raw generators:')
print('-' * 60)

async def aggregator_demo():
    token_list = []
    async for token in agg.merge():
        print(f'  [{token.source:<10}] {token.text!r}')
        token_list.append(token)
    print(f'\nTotal merged tokens: {len(token_list)}')
    by_source = {}
    for t in token_list:
        by_source.setdefault(t.source, []).append(t.text)
    for src, parts in by_source.items():
        print(f'  {src}: "{"".join(parts).strip()}"')

await aggregator_demo()

In [ ]:
# StreamAggregator also handles pre-formed StreamToken objects
async def typed_stream(ctx):
    for i, word in enumerate(['Alpha', 'Beta', 'Gamma']):
        await asyncio.sleep(0.02)
        yield StreamToken(text=word + ' ', source='typed', is_final=(i == 2))

agg2 = StreamAggregator()
agg2.add('typed_branch', typed_stream({}))

async def typed_demo():
    async for token in agg2.merge():
        print(f'  token: {token.text!r}  source={token.source}  final={token.is_final}')

await typed_demo()

## Section 4: Partial Result Bus

`PartialResultBus` is a synchronous pub/sub bus for incremental result events.
Subscribers register with a `run_id` (exact match) or `"*"` for all runs.

Use it for:
- Live UI progress updates
- Logging intermediate agent outputs
- Triggering downstream steps as soon as partial data is ready

In [ ]:
bus = PartialResultBus()

# Track events received
run_42_events = []
all_events    = []

# Subscribe to a specific run
bus.subscribe('run-42', lambda pr: run_42_events.append(pr))

# Subscribe to ALL runs with a wildcard
bus.subscribe('*', lambda pr: all_events.append(pr))

print('Bus created, subscribers registered')
print(f'Registered run_ids: {bus.run_ids()}')

In [ ]:
# Emit a series of partial results simulating a multi-step workflow
steps = [
    ('parse',    'Parsed 1,024 tokens from input.',   0.10),
    ('retrieve', 'Retrieved 5 relevant documents.',   0.35),
    ('reason',   'Generated chain-of-thought draft.', 0.65),
    ('refine',   'Applied self-critique pass.',        0.85),
    ('output',   'Final answer ready.',                1.00),
]

for step_name, content, progress in steps:
    pr = PartialResult(
        run_id='run-42',
        step=step_name,
        content=content,
        progress=progress,
        is_final=(progress == 1.0),
    )
    bus.emit('run-42', pr)
    print(f'  [{progress:.0%}] {step_name}: {content}')

print(f'\nrun-42 subscriber received: {len(run_42_events)} events')
print(f'wildcard (*) subscriber received: {len(all_events)} events')

In [ ]:
# Emit events from a second run — only wildcard subscriber should see them
for step_name, content, progress in steps[:3]:
    pr = PartialResult(
        run_id='run-99',
        step=step_name,
        content=content,
        progress=progress,
    )
    bus.emit('run-99', pr)

print(f'After emitting run-99 events:')
print(f'  run-42 subscriber still has: {len(run_42_events)} events (unchanged)')
print(f'  wildcard (*) subscriber now has: {len(all_events)} events (run-42 + run-99)')

In [ ]:
# Inspect the captured events
print('Captured run-42 events:')
for ev in run_42_events:
    status = 'FINAL' if ev.is_final else '     '
    print(f'  [{status}] step={ev.step:<10} progress={ev.progress:.0%}  content={ev.content!r}')

In [ ]:
# Simulating a live streaming agent that emits partial results while streaming tokens
async def streaming_agent_with_bus(run_id: str, prompt: str):
    bus2 = PartialResultBus()
    live_display = []
    bus2.subscribe(run_id, lambda pr: live_display.append(f'[{pr.progress:.0%}] {pr.content}'))

    async def word_stream(ctx):
        words = ctx['prompt'].split()
        for i, w in enumerate(words):
            await asyncio.sleep(0.02)
            # Emit a partial result every 3 words
            if (i + 1) % 3 == 0 or i == len(words) - 1:
                bus2.emit(run_id, PartialResult(
                    run_id=run_id,
                    step='generate',
                    content=' '.join(words[:i+1]),
                    progress=(i + 1) / len(words),
                    is_final=(i == len(words) - 1),
                ))
            yield w + ' '

    sa = StreamingAgent('live_agent', stream_fn=word_stream)
    full_text = await sa.run({'prompt': prompt})
    return full_text, live_display

full, snapshots = await streaming_agent_with_bus(
    'run-live',
    'AI systems can stream partial results as they generate each token',
)

print('Live progress snapshots:')
for snap in snapshots:
    print(f'  {snap}')
print(f'\nFinal full text: "{full.strip()}"')

## Summary

```python
# Minimal streaming pipeline
from multigen.streaming import StreamingAgent, ParallelStreamer, PartialResultBus, PartialResult

# 1. Single streaming agent
agent = StreamingAgent('writer', stream_fn=my_llm_stream)
async for token in agent.stream({'prompt': 'Write a poem'}):
    print(token.text, end='', flush=True)

# 2. Parallel fan-out
ps = ParallelStreamer([agent_a, agent_b, agent_c])
async for token in ps.stream(ctx):
    print(f'[{token.source}] {token.text}')

# 3. Progress events
bus = PartialResultBus()
bus.subscribe('run-1', lambda pr: update_ui(pr.progress, pr.content))
bus.emit('run-1', PartialResult(run_id='run-1', step='step1', content='...', progress=0.5))
```

**Key properties:**
- `StreamToken.source` — identifies which agent produced the token
- `StreamToken.is_final` — True on the last token from a stream
- `PartialResult.progress` — float 0.0-1.0 for progress bars
- `PartialResultBus` subscribers receive events synchronously in the order they were registered